# Atelier ML - Apprentissage: Parties 1 et 2
## But principal: choisir le bon Pokemon

**Probleme a resoudre:** apprendre le dresseur le bon Pokemon a utiliser lors d'un combat

Etapes du projet de Machine Learning:
1. Définition du problème à résoudre
2. Acquisition des données d’apprentissages et de tests
3. Préparer et nettoyer les données
4. Analyser, explorer les données
5. Choisir un modèle d’apprentissage
6. Visualiser les résultats, et ajuster ou modifier le modèle d’apprentissage

## Librairies utilisees: 

1. Numpy: Permet la manipulation de tableaux et matrices
2. Pandas: Permet la manipulation et l'analyse de donnees

In [ ]:
import numpy as nmp
import pandas as pd

pd.set_option('display.max_columns', None) #Afficher toutes les colonnes
pd.set_option('future.no_silent_downcasting', True) #Supprimer les avertissements de downcasting

# 1. Preparation des donnees
## Les donnees du Pokedex

Les fichiers `pokedex.csv` et `combats.csv` sont situes dans le repertoire `data`.  On utilise `read_csv` pour charger le Pokedex dans un Dataframe.

In [ ]:
nosPokemons = pd.read_csv("data/pokedex.csv")

# Affichage des colonnes du DataFrame
print(nosPokemons.columns.values)

| Colonne | Signification |
|---|---|
| NUMERO | Identifiant du Pokémon |
| NOM | Nom du Pokémon |
| TYPE_1 | Type primaire |
| TYPE_2 | Type secondaire |
| POINTS_DE_VIE | Points de vie |
| NIVEAU_ATTAQUE | Niveau d'attaque |
| NIVEAU_DEFENSE | Niveau de défense |
| NIVEAU_ATTAQUE_SPECIALE | Niveau d'attaque spéciale |
| NIVEAU_DEFENSE_SPECIALE | Niveau de défense spéciale |
| VITESSE | Vitesse |
| GENERATION | Génération du Pokémon |
| LEGENDAIRE | Pokémon légendaire (booléen) |

In [ ]:
# Utilisation de Head du Dataframe pour afficher les dix premieres lignes

print(nosPokemons.head(10))

Chaque ligne est une **observation** et chaque colonne une **feature**.

## Classement des features
- **Categorisation categorielle** : `TYPE_1`, `TYPE_2`, `LEGENDAIRE`
- **Categorisation numerique discrete** : `NUMERO`, `POINTS_DE_VIE`, `NIVEAU_ATTAQUE`, `NIVEAU_DEFENSE`, `NIVEAU_ATTAQUE_SPECIALE`, `NIVEAU_DEFENSE_SPECIALE`, `VITESSE`
- **Categorisation ordinale** : `GENERATION`

### Feature LEGENDAIRE
Le Machine Learning travaille seulement avec les donnees numeriques, donc "VRAI" et "FAUX" n'est pas exploitable. Il faudra alors transformer VRAI en 1 et FAUX en 0

In [ ]:
# replacement de VRAI FAUX par 1 0 
nosPokemonsClean = nosPokemons
nosPokemonsClean['LEGENDAIRE'] = nosPokemonsClean['LEGENDAIRE'].replace("VRAI", 1)
nosPokemonsClean['LEGENDAIRE'] = nosPokemonsClean['LEGENDAIRE'].replace("FAUX", 0)
print(nosPokemonsClean.head())

### Manque-t-il des donnees ?
On compte les observations et les features avec `shape`, puis on cherche les valeurs absentes avec `info()`.

In [ ]:
#Comptage du nombre d'observations et de features
print(nosPokemonsClean.shape)

In [ ]:
#Informations sur notre jeu de données
print(nosPokemonsClean.info())

**Constats :**
- 800 observations et 12 features
- Il manque 1 `NOM` (799 sur 800)
- Il manque 387 `TYPE_2` (413 sur 800). C'est normal : un Pokemon peut ne pas avoir de type secondaire.

### Le nom manquant
On ne peut pas savoir si le nom a une incidence sur le résultat des combats, donc on ne laisse pas l'observation incomplete. On commence par la trouver.

In [ ]:
print(nosPokemonsClean[nosPokemonsClean['NOM'].isnull()])

Il s'agit de l'observation 62. Les Pokémons sont dans l'ordre du Pokedex national : on regarde le nom du precedent et du suivant, puis on cherche dans le Pokedex ([Pokepedia](https://www.pokepedia.fr/Liste_des_Pok%C3%A9mon_dans_l%27ordre_du_Pok%C3%A9dex_National)).

In [ ]:
print(nosPokemonsClean['NOM'][61])
print(nosPokemonsClean['NOM'][63])

Le Pokemon entre Férosinge et Caninos est **Colossinge**. On renseigne la feature `NOM` de l'observation 62.

In [ ]:
nosPokemonsClean.loc[62, 'NOM'] = "Colossinge"
print(nosPokemonsClean['NOM'][62])

Cette modification ne touche que le Dataframe, pas le fichier `pokedex.csv` d'origine.

## Les donnees des combats
On applique la meme demarche a `combats.csv` : chargement, features, dix premieres observations, comptage et recherche de donnees absentes.

In [ ]:
#Chargement des données des combats
combats = pd.read_csv("data/combats.csv")

#Affichage des colonnes du Dataframe
print(combats.columns.values)

#Affichage des 10 premières lignes du Dataframe
print(combats.head(10))

#Comptage du nombre de lignes et de colonnes
print (combats.shape)

#Informations sur notre jeu de données
print (combats.info())

**Constats :**
- 50 000 observations et 3 features : `Premier_Pokemon`, `Second_Pokemon`, `Pokemon_Gagnant`.
- `Pokemon_Gagnant` contient le numero du Pokemon gagnant
- Chaque feature est de type numerique
- Il ne manque aucune information

## Assemblage des observations
Il faut reunir les informations des combats et des caracteristiques des Pokemons afin de savoir pour chaque Pokemon le nombre de combats menes ainsi que le nombre de victoires

### Nombre de combats menes
Lors d’un combat, le Pokémon peut être utilisé en première position ou en seconde position. Cette position à une importance car elle permet de savoir qui va donner le premier coup. Dans le cas où notre pokemon à peu de points de vie, s’il joue en seconde position, on risque de perdre avant de donner notre premier coup

| Combat | Premier Pokemon | Second Pokemon |
|---|---|---|
| 1 | Bulbizarre | Dracaufeu |
| 2 | Dracaufeu | Colossinge |
| 3 | Dracaufeu | Pikachu |

Donnant alors le nombre de combats par Pokémon :

| Pokemon | Nombre de combats |
|---|---|
| Bulbizarre | 1 |
| Dracaufeu | 3 |
| Colossinge | 1 |
| Pikachu | 1 |

In [ ]:
nbFoisPremierePosition = combats.groupby('Premier_Pokemon').count() # Comptage du nombre de fois où chaque Pokémon apparaît en première position
print(nbFoisPremierePosition) 

Si l’on reprend notre exemple, les données que nous obtenons seraient :
| Pokemon | Nombre PREMIERE_POSITION |
|---|---|
| Bulbizarre | 1 |
| Dracaufeu | 2 |
| Colossinge | 0 |
| Pikachu | 0 |

Il s’agit donc de compter le nombre de fois où le Pokémon est en première position via l’instruction `count()` puis de regrouper le résultat via l’instruction `groupBy()`

In [ ]:
nbFoisSecondePosition = combats.groupby('Second_Pokemon').count()
print(nbFoisSecondePosition)

nombreTotalDeCombats = nbFoisPremierePosition + nbFoisSecondePosition
print(nombreTotalDeCombats)

### Nombre de combats gagnés

In [ ]:
nombreDeVictoires = combats.groupby('Pokemon_Gagnant').count()
print(nombreDeVictoires)

En agrégeant les deux tableaux que nous venons de constituer, on constate que le Pokémon 228 a gagné 104 combats, alors qu’il en a mené 150 (cf. tableau précédent) alors que les quatre premiers Pokémons ont gagné tous leurs combats

En utilisant la fonction `info()`, nous constatons que nous disposons que de 783 observations. Cela signifie qu’il existe des Pokémons n’ayant jamais gagné ! Charge à présent de les découvrir

In [ ]:
jamaisGagne = nosPokemonsClean[~nosPokemonsClean['NUMERO'].isin(nombreDeVictoires.index)]
print(len(jamaisGagne), "Pokémons n'ont jamais gagné :")
print(jamaisGagne[['NUMERO', 'NOM', 'TYPE_1', 'POINTS_DE_VIE', 'NIVEAU_ATTAQUE', 'VITESSE']])

### Agrégation des données avec le Pokédex
On crée d'abord une liste contenant, pour chaque numéro de Pokémon : le nombre de combats, le nombre de victoires et le pourcentage de victoires. Le numéro sert de clé d'agrégation avec le Pokédex.

In [ ]:
listeAAgreger = combats.groupby('Pokemon_Gagnant').count()
listeAAgreger.sort_index()

#On ajoute le nombre de combats
listeAAgreger['NBR_COMBATS'] = nbFoisPremierePosition.Pokemon_Gagnant + nbFoisSecondePosition.Pokemon_Gagnant

#On ajoute le nombre de victoires
listeAAgreger['NBR_VICTOIRES'] = nombreDeVictoires.Premier_Pokemon

#On calcule le pourcentage de victoires
listeAAgreger['POURCENTAGE_DE_VICTOIRES'] = nombreDeVictoires.Premier_Pokemon / (nbFoisPremierePosition.Pokemon_Gagnant + nbFoisSecondePosition.Pokemon_Gagnant)

#On affiche la nouvelle liste
print(listeAAgreger)

On fusionne avec le Pokédex grâce à `merge()`, en utilisant l'index de la liste (`right_index=True`) lié à la feature `NUMERO` (`how='left'` conserve les 800 Pokémons, y compris ceux qui n'ont jamais gagné).

In [ ]:
nouveauPokedex = nosPokemonsClean.merge(listeAAgreger, left_on='NUMERO', right_index = True, how='left')
print(nouveauPokedex.describe())

# 2. Analyses statistiques

## Un peu de statistiques
Avant d’entrer dans le vif du sujet, nous allons faire un peu de statistiques sur notre jeu d’observations à l’aide de la fonction `describe()` du module Pandas.

In [ ]:
print(nouveauPokedex.describe())

À titre d’exemple, prenons le résultat de l’analyse statistique de la feature NIVEAU_ATTAQUE:
- **count** = 800: aucune donnee manquante
- **mean** (moyenne) = 79: on peut considerer qu'en dessous de cette moyenne, le Pokemon a un niveau faible, et au-dessus, il a un niveau fort. Premiere information interessantes
- **std** (ecart-type) = 32:  les données de la feature NIVEAU_ATTAQUE semblent être assez éloignées de la moyenne. Cela signifie qu’il existe de nombreux cas différents de niveaux d’attaque et que les observations ne sont pas homogènes. C’est un point que nous devrons vérifier lors de la visualisation des données
- **min = 5, max = 190** : de très faibles à très forts niveaux d'attaque.
- **Quartiles** : 25 % des Pokémons ont une attaque inférieure à 55, 50 % inférieure à 75, 75 % inférieure à 100. Peu de Pokémons (25 %) dépassent 100.

### Description du jeu d'observations